Robust system path setup for project modules


In [6]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


 Import Core Libraries and GEE Initialization 

In [7]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee # Google Earth Engine API
import pandas as pd # Data manipulation
import geopandas as gpd # Geospatial data manipulation (builds on pandas)
import folium # For interactive mapping in notebooks
from configs.regions import kenyan_coast_roi # Import your defined region from config

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') 

print("All core libraries imported and GEE initialized.")

All core libraries imported and GEE initialized.


Importing the carbon coefficients


In [8]:
# Cell 3: Import Carbon Coefficients
from configs.carbon_coefficients import (
    CARBON_FRACTION_BIOMASS,
    BGB_AGB_RATIO,
    SOIL_CARBON_DENSITY_PER_M,
    DEFAULT_SOIL_DEPTH_M
)

print("Carbon coefficients imported successfully.")
print(f"Example: Carbon Fraction of Biomass = {CARBON_FRACTION_BIOMASS}")

Carbon coefficients imported successfully.
Example: Carbon Fraction of Biomass = 0.47


Fetch GMW Mangrove Extent & GEDI Biomass Data (from other notebooks)


In [9]:
# Cell 4: Fetch GMW Mangrove Extent & GEDI Biomass Data
print("--- Fetching GMW and GEDI Data ---")

# --- GMW Mangrove Extent ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")
mangroves_in_roi = recent_mangrove_image.clip(kenyan_coast_roi)
mangrove_mask = mangroves_in_roi.gt(0)
mangroves_filtered = mangroves_in_roi.updateMask(mangrove_mask)
print("GMW Mangrove Extent loaded.")

# --- GEDI L4A Biomass ---
gedi_collection_id = "LARSE/GEDI/GEDI04_A_002_MONTHLY"
gedi_data = ee.ImageCollection(gedi_collection_id)
start_date = '2019-04-01'
end_date = '2023-12-31'
gedi_filtered = gedi_data.filterDate(start_date, end_date)
gedi_in_roi_collection = gedi_filtered.filterBounds(kenyan_coast_roi)

num_gedi_images = gedi_in_roi_collection.size().getInfo()
if num_gedi_images == 0:
    print("WARNING: No GEDI L4A data found for carbon calculation. Using dummy image.")
    gedi_agbd_image = ee.Image(0).rename('agbd').clip(kenyan_coast_roi)
else:
    gedi_agbd_image = gedi_in_roi_collection.median().select('agbd').clip(kenyan_coast_roi)
print("GEDI L4A Biomass loaded.")

# Apply the mangrove mask to the GEDI AGBD image
# We only want to estimate carbon where there are mangroves.
mangrove_agbd = gedi_agbd_image.updateMask(mangroves_filtered)

--- Fetching GMW and GEDI Data ---
GMW Mangrove Extent loaded.
GEDI L4A Biomass loaded.


In [ ]:
# T0 debug band names
print("Bands available in recent_mangrove_image:")
print(recent_mangrove_image.bandNames().getInfo())

Bands available in recent_mangrove_image:
['1']


Calculate Aboveground Carbon (AGC) Stock


In [17]:
# Cell 5: Calculate Aboveground Carbon (AGC) Stock
print("--- Calculating Aboveground Carbon (AGC) Stock ---")

# AGBD (Aboveground Biomass Density) from GEDI is typically in Mg/ha (megagrams per hectare)
# 1 Mg = 1 tonne. So, AGBD is in tonnes per hectare.

# Calculate Aboveground Carbon Density (AGC in tonnes C / hectare)
# AGC = AGBD * CARBON_FRACTION_BIOMASS
agc_density_image = mangrove_agbd.multiply(CARBON_FRACTION_BIOMASS).rename('AGC_Density_tonnes_C_ha')

# Now, let's get the total area of mangroves within our ROI.
# The 'mangroves_filtered' image has 1 where mangroves exist, 0 otherwise.
# Reduce this image to get pixel areas.
# Pixel area in square meters. Divide by 10000 to get hectares.
pixel_area_ha = ee.Image.pixelArea().divide(10000) # Each pixel's area in hectares

# Calculate total AGC in tonnes of Carbon
# Sum (AGC_Density_tonnes_C_ha * pixel_area_ha) over the ROI
total_agc_tonnes = agc_density_image.multiply(pixel_area_ha) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=kenyan_coast_roi,
        scale=30, # Use an appropriate scale (e.g., 30m for Landsat-derived data)
        maxPixels=1e10 # Increase maxPixels for larger areas
    ).get('AGC_Density_tonnes_C_ha') # Get the sum for the 'AGC_Density_tonnes_C_ha' band

print(f"Calculated Total Aboveground Carbon (AGC) Stock in ROI: {total_agc_tonnes.getInfo():,.2f} tonnes C")

# Optional: Calculate total mangrove area in hectares for context
mangrove_area_image = mangroves_filtered.multiply(pixel_area_ha)
total_mangrove_area_ha = mangrove_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=kenyan_coast_roi,
    scale=30,
    maxPixels=1e10
).get('1').getInfo() # 'classification' is default band name for LANDSAT/MANGROVE_FORESTS output

print(f"Total Mangrove Area in ROI: {total_mangrove_area_ha:,.2f} hectares")

--- Calculating Aboveground Carbon (AGC) Stock ---
Calculated Total Aboveground Carbon (AGC) Stock in ROI: 25,444.17 tonnes C
Total Mangrove Area in ROI: 31,676.78 hectares


Visualize Aboveground Carbon (AGC) Density & Export Raster


In [16]:
# Cell 6: Visualize Aboveground Carbon (AGC) Density & Prepare for Export (REVISED PROJECT ID ACCESS)
print("--- Visualizing AGC Density ---")

# Define visualization parameters for AGC Density
agc_vis_params = {
    'min': 0,
    'max': 150, # Example max, adjust based on expected range
    'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494']
}

# Re-initialize Folium map
centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]
m_agc = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

# Add ROI GeoJSON layer
roi_geojson = kenyan_coast_roi.getInfo()
folium.GeoJson(
    roi_geojson,
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}
).add_to(m_agc)

# Add AGC Density raster layer
map_id_dict_agc = agc_density_image.getMapId(agc_vis_params)
folium.TileLayer(
    tiles=map_id_dict_agc['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Aboveground Carbon Density (tonnes C/ha)'
).add_to(m_agc)

folium.LayerControl().add_to(m_agc)
m_agc

# --- Prepare for Export to GEE Assets ---
# REPLACED ee.data.getProjectName() with the direct project ID
# Ensure 'gaias-ark' below is replaced with YOUR ACTUAL GEE PROJECT ID
my_gee_project_id = 'gaias-ark' 

output_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_agc_density_kenya'
output_description = 'Aboveground Carbon Density for Kenyan Coast'

task = ee.batch.Export.image.toAsset(
    image=agc_density_image.unmask(0),
    description=output_description,
    assetId=output_asset_id,
    scale=30,
    region=kenyan_coast_roi.bounds(),
    maxPixels=1e10
)

task.start()
print(f"\nGEE Export Task initiated for AGC Density. Asset ID: {output_asset_id}")
print("Check the 'Tasks' tab in your GEE Code Editor to monitor progress.")

--- Visualizing AGC Density ---

GEE Export Task initiated for AGC Density. Asset ID: projects/gaias-ark/assets/gaias_ark_agc_density_kenya
Check the 'Tasks' tab in your GEE Code Editor to monitor progress.
